# 🧭 CDM Compass
**Clinical Data Management Knowledge Navigator**

Ask questions about your CDM/CDS documents in plain English and get answers with inline citations.
Documents are classified by folder. Regulatory requirements and opinion papers are treated differently.
The interactive loop (Block 8) remembers your last few questions within a session.

### 📋 How to use this notebook
Run each block **from top to bottom**, in order. Re-run from Block 3 onwards after adding new documents.

| Block | What it does | Run again? |
|-------|-------------|------------|
| 1 | Install packages | Only once |
| 2 | Import libraries | Each session |
| 3 | Load documents | When you add files |
| 4 | Chunk text | When you add files |
| 5 | Build search index | When you add files |
| 6 | Connect to Ollama LLM | Each session |
| 7 | Ask a question | Anytime! |
| 8 | Interactive Q&A loop (with memory) | Anytime! |

### 📁 Document authority
```
data/raw/
├── regulatory/    <- binding requirements (ICH, FDA, EMA)
└── opinion/       <- position papers, white papers (SCDM, JSCDM)
```
Files placed anywhere else will raise an error.


## 📦 Block 1 -- Install Required Packages
Run this once. It installs everything CDM Compass needs.

> ⚠️ **Before running Block 6**, install Ollama separately:
> 1. Go to [https://ollama.com](https://ollama.com) and download the app
> 2. Open Terminal and run: `ollama pull qwen2.5:14b`
> 3. Ollama runs in the background automatically


In [1]:
# Install all required packages -- run this once
import sys

!{sys.executable} -m pip install --upgrade pip --quiet
!{sys.executable} -m pip install "sentence-transformers[cross-encoder]" chromadb pypdf openpyxl python-docx python-pptx rank-bm25 requests tqdm --quiet
!{sys.executable} -m pip install --upgrade jupyter ipywidgets --quiet

print("\u2705 All packages installed successfully!")


✅ All packages installed successfully!


## 📚 Block 2 -- Import Libraries
Loads all tools into memory. Run this at the start of every session.

All imports live here -- no other block adds imports.


In [2]:
import os
import re
import json
import csv
import textwrap
import xml.etree.ElementTree as ET
import requests
import numpy as np
import torch
import chromadb
from rank_bm25 import BM25Okapi

from pypdf import PdfReader
from pptx import Presentation
from sentence_transformers import SentenceTransformer, CrossEncoder
from tqdm.notebook import tqdm
import openpyxl
import docx  # python-docx

print("\u2705 All libraries imported successfully!")


✅ All libraries imported successfully!


## 📁 Block 3 — Load Documents

**👇 Only change this line** if your documents folder is in a different location.

Documents must be placed in one of two subfolders — this determines how they are treated:

| Subfolder | Authority | Use for |
|-----------|-----------|---------|
| `regulatory/` | ⚖️ Regulatory | ICH, FDA, EMA — binding requirements |
| `opinion/` | 💬 Opinion | SCDM, JSCDM, white papers, position papers |

**Supported file types:** PDF, PPTX, XLSX, DOCX, JSON, XML, CSV, TXT

> ⚠️ Old-format `.ppt` files are not supported. Open in PowerPoint → File → Save As → .pptx first.


In [3]:
# ============================================================
# 🔴 CHANGE THIS to your documents folder path if needed
# ============================================================
DOCS_FOLDER = "../data/raw/"
# ============================================================
#
# Authority is set automatically by subfolder:
#   data/raw/regulatory/  → binding requirements (ICH, FDA, EMA)
#   data/raw/opinion/     → position papers and recommendations
#
# Files found outside these two folders will raise an error.
# ============================================================

AUTHORITY_FOLDERS = {"regulatory": "regulatory", "opinion": "opinion"}


def get_authority(filepath):
    """Derive authority level from the subfolder the file lives in."""
    parts = filepath.replace("\\", "/").split("/")
    for part in parts:
        if part.lower() in AUTHORITY_FOLDERS:
            return AUTHORITY_FOLDERS[part.lower()]
    raise ValueError(
        f"\n❌ Cannot classify: '{os.path.basename(filepath)}'\n"
        f"   Move it into one of these subfolders before loading:\n"
        f"     {DOCS_FOLDER}regulatory/   ← binding requirements (ICH, FDA, EMA)\n"
        f"     {DOCS_FOLDER}opinion/       ← position papers and white papers"
    )


def extract_heading(text):
    """Try to find a heading on the page (short ALL-CAPS line)."""
    for line in text.split("\n"):
        stripped = line.strip()
        if 4 < len(stripped) < 80 and stripped.isupper():
            return stripped
    return "Unknown"


def load_pdf(filepath):
    """Load a PDF file. Returns one entry per page."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        reader = PdfReader(filepath)
        for i, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and text.strip():
                docs.append({"text": text.strip(), "source": filename,
                              "page": i + 1, "heading": extract_heading(text),
                              "authority": authority})
    except Exception as e:
        print(f"  ❌ Error reading PDF {filename}: {e}")
    return docs


def load_pptx(filepath):
    """Load a PowerPoint file. Extracts text from slides and speaker notes."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        prs = Presentation(filepath)
        for i, slide in enumerate(prs.slides, 1):
            parts = [shape.text_frame.text.strip()
                     for shape in slide.shapes if shape.has_text_frame]
            if slide.has_notes_slide:
                notes = slide.notes_slide.notes_text_frame.text.strip()
                if notes:
                    parts.append(f"[Speaker notes] {notes}")
            text = "\n".join(p for p in parts if p)
            if text:
                docs.append({"text": text, "source": filename,
                              "page": i, "heading": f"Slide {i}",
                              "authority": authority})
    except Exception as e:
        print(f"  ❌ Error reading PPTX {filename}: {e}")
    return docs


def load_xlsx(filepath):
    """Load an Excel file. Returns one entry per sheet."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            rows_text = [" | ".join(str(c) for c in row if c is not None)
                         for row in ws.iter_rows(values_only=True)]
            rows_text = [r for r in rows_text if r.strip()]
            if rows_text:
                docs.append({"text": "\n".join(rows_text), "source": filename,
                              "page": sheet_name, "heading": f"Sheet: {sheet_name}",
                              "authority": authority})
    except Exception as e:
        print(f"  ❌ Error reading XLSX {filename}: {e}")
    return docs


def load_docx(filepath):
    """Load a Word document."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        document = docx.Document(filepath)
        paragraphs = [p.text.strip() for p in document.paragraphs if p.text.strip()]
        full_text  = "\n".join(paragraphs)
        if full_text:
            docs.append({"text": full_text, "source": filename, "page": 1,
                          "heading": paragraphs[0][:80] if paragraphs else "Unknown",
                          "authority": authority})
    except Exception as e:
        print(f"  ❌ Error reading DOCX {filename}: {e}")
    return docs


def load_json(filepath):
    """Load a JSON file."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        docs.append({"text": json.dumps(data, indent=2, ensure_ascii=False),
                     "source": filename, "page": 1, "heading": "JSON Document",
                     "authority": authority})
    except Exception as e:
        print(f"  ❌ Error reading JSON {filename}: {e}")
    return docs


def load_xml(filepath):
    """Load an XML file."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        root = ET.parse(filepath).getroot()
        texts = [f"{e.tag}: {e.text.strip()}" for e in root.iter()
                 if e.text and e.text.strip()]
        full_text = "\n".join(texts)
        if full_text:
            docs.append({"text": full_text, "source": filename, "page": 1,
                          "heading": f"XML Root: {root.tag}", "authority": authority})
    except Exception as e:
        print(f"  ❌ Error reading XML {filename}: {e}")
    return docs


def load_csv(filepath):
    """Load a CSV file."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        with open(filepath, "r", encoding="utf-8", errors="replace") as f:
            rows = [" | ".join(row) for row in csv.reader(f)
                    if any(c.strip() for c in row)]
        full_text = "\n".join(rows)
        if full_text:
            docs.append({"text": full_text, "source": filename, "page": 1,
                          "heading": "CSV Document", "authority": authority})
    except Exception as e:
        print(f"  ❌ Error reading CSV {filename}: {e}")
    return docs


def load_txt(filepath):
    """Load a plain text file."""
    docs, filename, authority = [], os.path.basename(filepath), get_authority(filepath)
    try:
        with open(filepath, "r", encoding="utf-8", errors="replace") as f:
            text = f.read().strip()
        if text:
            docs.append({"text": text, "source": filename, "page": 1,
                          "heading": "Text Document", "authority": authority})
    except Exception as e:
        print(f"  ❌ Error reading TXT {filename}: {e}")
    return docs


LOADERS = {
    ".pdf":  load_pdf,  ".pptx": load_pptx,
    ".xlsx": load_xlsx, ".xls":  load_xlsx,
    ".docx": load_docx, ".json": load_json,
    ".xml":  load_xml,  ".csv":  load_csv,
    ".txt":  load_txt,
}


def load_all_documents(folder_path):
    """Walk the folder tree and load all supported files."""
    all_docs, skipped = [], []

    if not os.path.exists(folder_path):
        print(f"❌ Folder not found: {folder_path}")
        print("   Update DOCS_FOLDER above to point to your documents folder.")
        return all_docs

    for root, dirs, files in os.walk(folder_path):
        for filename in sorted(files):
            ext       = os.path.splitext(filename)[1].lower()
            full_path = os.path.join(root, filename)

            if ext == ".ppt":
                print(f"  ⚠️  Skipped (old format): {filename}")
                print(f"     → Open in PowerPoint and save as .pptx, then re-add.")
                skipped.append(filename)
                continue

            if ext in LOADERS:
                try:
                    print(f"  📄 Loading: {filename}")
                    docs = LOADERS[ext](full_path)
                    auth = get_authority(full_path).upper()
                    print(f"     → {len(docs)} section(s)  [{auth}]")
                    all_docs.extend(docs)
                except ValueError as e:
                    print(e)
                    print("\n⛔ Loading stopped. Classify the file above and re-run Block 3.")
                    return all_docs
            elif not filename.startswith("."):
                skipped.append(filename)

    print(f"\n✅ Total sections loaded: {len(all_docs)}")
    if skipped:
        print(f"⚠️  Skipped: {', '.join(skipped)}")
    return all_docs


print(f"🔍 Scanning folder: {DOCS_FOLDER}\n")
docs = load_all_documents(DOCS_FOLDER)


🔍 Scanning folder: ../data/raw/

  📄 Loading: LOINC Public Review Comments.xlsx
     → 2 section(s)  [OPINION]
  📄 Loading: LOINC to LB Mapping-Read Me.pdf
     → 4 section(s)  [OPINION]
  📄 Loading: LOINC_to_LB_Mapping Document_FINAL.csv
     → 1 section(s)  [OPINION]
  📄 Loading: LOINC_to_LB_Mapping Document_FINAL.xlsx
     → 3 section(s)  [OPINION]
  📄 Loading: Guidance for eCOA Development in Clinical Trials.pdf
     → 23 section(s)  [OPINION]
  📄 Loading: jscdm-116-lebedys.pdf
     → 20 section(s)  [OPINION]
  📄 Loading: jscdm-117-hills.pdf
     → 12 section(s)  [OPINION]
  📄 Loading: jscdm-118-amatya.pdf
     → 13 section(s)  [OPINION]
  📄 Loading: jscdm-20-stokman.pdf
     → 8 section(s)  [OPINION]
  📄 Loading: jscdm-260-king.pdf
     → 9 section(s)  [OPINION]
  📄 Loading: jscdm-262-king.pdf
     → 10 section(s)  [OPINION]
  📄 Loading: jscdm-263-king.pdf
     → 10 section(s)  [OPINION]
  📄 Loading: jscdm-264-king.pdf
     → 7 section(s)  [OPINION]
  📄 Loading: jscdm-29-pestronk.

Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)


     → 14 section(s)  [OPINION]
  📄 Loading: The Uncomfortable Conversation AI And Data Use In Clinical Trials.pdf
     → 3 section(s)  [OPINION]
  📄 Loading: 45966642fnl.pdf
     → 15 section(s)  [REGULATORY]
  📄 Loading: 58687461fnlrv8_1.pdf
     → 23 section(s)  [REGULATORY]
  📄 Loading: ectd_tech_guide_v1.9.pdf
     → 31 section(s)  [REGULATORY]
  📄 Loading: fda-dscv11.0.xlsx
     → 6 section(s)  [REGULATORY]
  📄 Loading: sdTCG-v6.1-December-2025-FINAL.pdf
     → 104 section(s)  [REGULATORY]
  📄 Loading: E11_R1_Addendum.pdf
     → 25 section(s)  [REGULATORY]
  📄 Loading: E2A_Guideline.pdf
     → 12 section(s)  [REGULATORY]
  📄 Loading: E9-R1_Step4_Guideline_2019_1203.pdf
     → 22 section(s)  [REGULATORY]
  📄 Loading: E9_Guideline.pdf
     → 39 section(s)  [REGULATORY]
  📄 Loading: ICH_E11A_Guideline_Step4_2024_0821.pdf
     → 33 section(s)  [REGULATORY]
  📄 Loading: ICH_E11A_Guideline_Step4_2024_0821.pptx
     → 24 section(s)  [REGULATORY]
  📄 Loading: ICH_E2B(R3)_EWG_IWG_Appendix

## ✂️ Block 4 — Split Text into Chunks

Long documents are split into smaller overlapping pieces so the search engine can find precise answers.

- `CHUNK_SIZE` = words per chunk. **300** gives tighter, more precise citations than 500.
- `CHUNK_OVERLAP` = words shared between consecutive chunks — prevents answers from being cut at a boundary.


In [4]:
# ============================================================
# Settings
# ============================================================
CHUNK_SIZE    = 300   # words per chunk (300 = more precise citations)
CHUNK_OVERLAP = 50    # words of overlap between chunks
# ============================================================

chunks   = []  # text of each chunk
metadata = []  # source, page, heading, authority for each chunk

for doc in docs:
    words = doc["text"].split()
    step  = CHUNK_SIZE - CHUNK_OVERLAP

    for i in range(0, len(words), step):
        chunk_text = " ".join(words[i : i + CHUNK_SIZE])
        if chunk_text.strip():
            chunks.append(chunk_text)
            metadata.append({
                "source":    doc["source"],
                "page":      doc["page"],
                "heading":   doc["heading"],
                "authority": doc["authority"],
            })

print(f"✅ Created {len(chunks)} chunks from {len(docs)} document sections")
print(f"   (chunk size: {CHUNK_SIZE} words, overlap: {CHUNK_OVERLAP} words)")


✅ Created 17147 chunks from 3624 document sections
   (chunk size: 300 words, overlap: 50 words)


## 🧠 Block 5 -- Build the Search Index

Builds three components:

| Component | Type | Role |
|-----------|------|------|
| **ChromaDB** | Semantic (vector) | Finds chunks with similar *meaning* |
| **BM25** | Keyword | Finds chunks with exact *words* |
| **CrossEncoder** | Reranker | Re-scores top candidates with higher precision |

**This may take a few minutes** on large document sets -- models are cached after the first run.


In [5]:
# ── Step 1: Load the embedding model ─────────────────────────────────────────
print("⏳ Loading embedding model (downloads once, then cached)...")
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"   Using device: {device}")

embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print("✅ Embedding model ready!")

# ── Step 2: Encode all chunks into embeddings ─────────────────────────────────
print(f"\n⏳ Encoding {len(chunks)} chunks into embeddings...")
embeddings = embed_model.encode(
    chunks,
    show_progress_bar=True,
    batch_size=32,
    convert_to_numpy=True,
)
embeddings = embeddings.astype("float32")
print(f"✅ Embeddings created: {embeddings.shape} (chunks × dimensions)")

# ── Step 3: Build ChromaDB semantic index ─────────────────────────────────────
print("\n⏳ Building semantic index (ChromaDB)...")
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection("rag_docs")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="rag_docs",
    metadata={"hnsw:space": "cosine"},
)

# Add in batches — ChromaDB has a max batch size of 5461
BATCH_SIZE = 5000
total      = len(chunks)

for start in range(0, total, BATCH_SIZE):
    end = min(start + BATCH_SIZE, total)
    collection.add(
        ids        = [str(i) for i in range(start, end)],
        embeddings = embeddings[start:end].tolist(),
        documents  = chunks[start:end],
        metadatas  = metadata[start:end],
    )
    print(f"   Indexed {end}/{total} chunks...")

print(f"✅ Semantic index ready with {collection.count()} vectors")

# ── Step 4: Build BM25 keyword index ─────────────────────────────────────────
print("\n⏳ Building keyword index (BM25)...")
tokenised_chunks = [chunk.lower().split() for chunk in chunks]
bm25_index       = BM25Okapi(tokenised_chunks)
print(f"✅ Keyword index ready over {len(chunks)} chunks")
print("\n🎉 Both indexes built — hybrid search is ready!")

# -- Step 5: Load the cross-encoder reranker --
print("\n\u23f3 Loading reranker (downloads once, then cached)...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2", max_length=512)
print("\u2705 Reranker ready!")
print("\n\U0001f389 CDM Compass is ready!")


⏳ Loading embedding model (downloads once, then cached)...
   Using device: mps


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model ready!

⏳ Encoding 17147 chunks into embeddings...


Batches:   0%|          | 0/536 [00:00<?, ?it/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Reranker ready!

🎉 CDM Compass is ready!


## 🤖 Block 6 -- Connect to Ollama (Local AI)

Connects to Ollama running locally.

**Recommended models:**

| Model | Pull command | Notes |
|-------|-------------|-------|
| `qwen2.5:14b` | `ollama pull qwen2.5:14b` | Best citation reliability -- recommended ✅ |
| `mistral` | `ollama pull mistral` | Good alternative |
| `llama3.2` | `ollama pull llama3.2` | Also solid |
| `phi3` | `ollama pull phi3` | Fastest -- least reliable on citations |


In [6]:
# ============================================================
# Settings
# ============================================================
OLLAMA_MODEL = "qwen2.5:14b"   # recommended -- better citation reliability
# Alternatives: mistral  |  llama3.2  |  phi3
OLLAMA_URL   = "http://localhost:11434/api/generate"
# ============================================================


def ask_ollama(prompt, model=OLLAMA_MODEL, temperature=0.3):
    """Send a prompt to Ollama and return the response text."""
    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model":   model,
                "prompt":  prompt,
                "stream":  False,
                "options": {"temperature": temperature},
            },
            timeout=120,
        )
        response.raise_for_status()
        return response.json().get("response", "").strip()

    except requests.exceptions.ConnectionError:
        return (
            "\u274c Cannot connect to Ollama.\n"
            "   Make sure Ollama is installed and running.\n"
            "   Download from: https://ollama.com\n"
            f"   Then in Terminal run: ollama pull {OLLAMA_MODEL}"
        )
    except Exception as e:
        return f"\u274c Error: {e}"


print(f"\U0001f517 Testing connection to Ollama (model: {OLLAMA_MODEL})...")
test_reply = ask_ollama("Reply with exactly: 'Ollama is working!'")
print(f"Response: {test_reply}")

if "working" in test_reply.lower() or "ollama" in test_reply.lower():
    print("\n\u2705 Ollama is connected and ready!")
else:
    print("\n\u26a0\ufe0f  Got a response -- Ollama seems to be running.")


🔗 Testing connection to Ollama (model: qwen2.5:14b)...
Response: Ollama is working!

✅ Ollama is connected and ready!


## 🔍 Block 7 -- Search + Ask a Question

CDM Compass uses a four-stage pipeline per question:

1. **Hybrid search** -- ChromaDB (semantic) + BM25 (keyword), fused by weight
2. **Reranking** -- cross-encoder selects the best `FINAL_K` chunks from `TOP_K` candidates
3. **Compression** -- only the most relevant sentences per chunk reach the LLM
4. **Context-aware answer** -- optional conversation history included so follow-up questions work

Sources appear in the order cited in the answer. Gaps in numbering (e.g. [1],[2],[4]) mean those chunks were retrieved but not used.

**👇 Change `MY_QUESTION` to whatever you want to ask.**


In [7]:
# ============================================================
# 👇 Type your question here
# ============================================================
MY_QUESTION = "What skills does an experienced Clinical Data Manager of 25+ years need to transition to Data Science?"
TOP_K           = 10   # candidates retrieved before reranking
FINAL_K         = 5    # chunks sent to LLM after reranking
HISTORY_TURNS   = 3    # how many previous Q&A turns to include (0 = off)
TOKEN_WARN_LIMIT = 3000  # warn if estimated prompt tokens exceed this
# ============================================================
SEMANTIC_WEIGHT = 0.7
KEYWORD_WEIGHT  = 0.3
# ============================================================


# -- ANSI colour helpers --
def _c(code, text): return f"\033[{code}m{text}\033[0m"
def green(t):   return _c('32;1', t)
def yellow(t):  return _c('33;1', t)
def red(t):     return _c('31;1', t)
def magenta(t): return _c('35;1', t)
def blue(t):    return _c('34;1', t)
def bold(t):    return _c('1',    t)
def dim(t):     return _c('2',    t)


def estimate_tokens(text):
    """Estimate token count using the 1 token ~ 4 chars heuristic (English prose)."""
    return len(text) // 4


def hybrid_search(question, top_k=10):
    """Retrieve candidates using semantic + keyword search fused by weight."""
    query_embedding  = embed_model.encode([question]).astype('float32')
    sem_results      = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
    )
    sem_scores = {
        int(idx): 1.0 - dist
        for idx, dist in zip(
            sem_results['ids'][0],
            sem_results['distances'][0],
        )
    }

    tokenised_query  = question.lower().split()
    bm25_raw         = bm25_index.get_scores(tokenised_query)
    ceiling          = np.percentile(bm25_raw[bm25_raw > 0], 95) if (bm25_raw > 0).any() else 1.0
    bm25_norm        = np.clip(bm25_raw / ceiling, 0.0, 1.0)
    bm25_top_indices = np.argsort(bm25_norm)[::-1][:top_k]
    bm25_scores      = {int(i): float(bm25_norm[i]) for i in bm25_top_indices}

    fused = []
    for i in set(sem_scores.keys()) | set(bm25_scores.keys()):
        s_score  = sem_scores.get(i, 0.0)
        k_score  = bm25_scores.get(i, 0.0)
        combined = SEMANTIC_WEIGHT * s_score + KEYWORD_WEIGHT * k_score
        fused.append({
            'text':         chunks[i],
            'source':       metadata[i]['source'],
            'page':         metadata[i]['page'],
            'heading':      metadata[i]['heading'],
            'authority':    metadata[i]['authority'],
            'hybrid_score': round(1.0 - combined, 3),
            'sem_score':    round(s_score, 3),
            'kw_score':     round(k_score, 3),
        })

    fused.sort(key=lambda x: (
        0 if x['authority'] == 'regulatory' else 1,
        x['hybrid_score'],
    ))
    return fused[:top_k]


def rerank(question, candidates, final_k=5):
    """Re-score candidates with cross-encoder; regulatory sources surface first."""
    pairs  = [(question, c['text']) for c in candidates]
    scores = reranker.predict(pairs)
    for c, score in zip(candidates, scores):
        c['rerank_score'] = float(score)
    candidates.sort(key=lambda x: (
        0 if x['authority'] == 'regulatory' else 1,
        -x['rerank_score'],
    ))
    return candidates[:final_k]


def compress(question, chunk_text, min_sentences=2):
    """Return only the sentences most relevant to the question."""
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', chunk_text) if s.strip()]
    if len(sentences) <= min_sentences:
        return chunk_text
    pairs  = [(question, s) for s in sentences]
    scores = reranker.predict(pairs)
    threshold = float(np.mean(scores))
    kept = [s for s, sc in zip(sentences, scores) if sc >= threshold]
    if len(kept) < min_sentences:
        top_idx = np.argsort(scores)[::-1][:min_sentences]
        kept    = [sentences[i] for i in sorted(top_idx)]
    return ' '.join(kept)


def build_prompt(question, retrieved_chunks, history=None):
    """
    Build the LLM prompt.
    - history: list of (question, answer) tuples from recent turns
    - No authority label in headers -- prevents inline leakage
    - Chunks compressed to relevant sentences only
    - Token estimate printed as a sanity check
    """
    seen, unique = set(), []
    for chunk in retrieved_chunks:
        key = (chunk['source'], chunk['page'])
        if key not in seen:
            seen.add(key)
            unique.append(chunk)

    context_parts = []
    for i, chunk in enumerate(unique, 1):
        fname           = os.path.basename(chunk['source'])
        compressed_text = compress(question, chunk['text'])
        context_parts.append(
            f"[{i}] {fname} -- page {chunk['page']}\n"
            f"{compressed_text}"
        )
    context = "\n\n".join(context_parts)

    # Build conversation history block (most recent turns first)
    history_block = ""
    if history:
        turns = []
        for q, a in history[-HISTORY_TURNS:]:
            # Truncate long answers to keep history compact
            a_short = a[:400] + '...' if len(a) > 400 else a
            turns.append(f"Q: {q}\nA: {a_short}")
        history_block = (
            "=== RECENT CONVERSATION ===\n"
            + "\n\n".join(turns)
            + "\n\n"
        )

    prompt = (
        "You are an expert navigator of Clinical Data Management (CDM) "
        "and Clinical Data Science (CDS) knowledge.\n\n"
        "Answer the question below using ONLY the document excerpts provided.\n"
        "If conversation history is present, use it only for context -- "
        "do not cite it as a source.\n\n"
        "IMPORTANT -- language rules based on document type:\n"
        "- Regulatory documents (ICH, FDA, EMA): binding requirements.\n"
        "  Use: must, is required to, mandates, is prohibited, shall.\n"
        "- Opinion documents (SCDM, JSCDM, position papers): recommendations.\n"
        "  Use: recommends, suggests, proposes, considers, advises.\n"
        "If a regulatory source conflicts with an opinion source, flag this explicitly.\n\n"
        "CRITICAL citation rules -- follow exactly:\n"
        "- Write citations as [1], [2], [3] only -- a number between square brackets, nothing else.\n"
        "- Never write [1: filename], [2, page X], or any other variant.\n"
        "- Place the bracket immediately after the claim, before any punctuation.\n"
        "- Do NOT list sources or references at the end of your answer.\n"
        "- Do NOT use ibid. -- repeat the bracket number every time.\n"
        "- If context is insufficient, say so clearly.\n\n"
        + history_block
        + "=== DOCUMENT CONTEXT ===\n"
        + context
        + "\n\n=== QUESTION ===\n"
        + question
        + "\n\n=== ANSWER ==="
    )
    return prompt


def ask_question(question, top_k=10, final_k=5, history=None):
    """CDM Compass pipeline: hybrid -> rerank -> compress -> answer -> cited sources."""
    print()
    print(bold(f'\U0001f9ed CDM Compass  --  "{question}"'))
    suffix = f' | {len(history)} turn(s) in memory' if history else ''
    print(dim(f'   Hybrid: {int(SEMANTIC_WEIGHT*100)}% semantic + {int(KEYWORD_WEIGHT*100)}% keyword  |  reranking top {final_k} of {top_k}{suffix}'))

    candidates = hybrid_search(question, top_k=top_k)
    retrieved  = rerank(question, candidates, final_k=final_k)
    print(dim(f'   {len(retrieved)} chunks after reranking'))

    prompt = build_prompt(question, retrieved, history=history)

    # Token budget warning
    est_tokens = estimate_tokens(prompt)
    if est_tokens > TOKEN_WARN_LIMIT:
        print(dim(f'   \u26a0\ufe0f  Estimated prompt size: ~{est_tokens} tokens (limit: {TOKEN_WARN_LIMIT}) -- consider reducing HISTORY_TURNS or FINAL_K'))
    else:
        print(dim(f'   Estimated prompt: ~{est_tokens} tokens'))

    print(dim(f'   Asking {OLLAMA_MODEL}...'))
    answer = ask_ollama(prompt)

    # Strip reference block the LLM appends
    answer = re.split(
        r'\n\s*(\[?[Rr]eferences?\]?[\s:]*$'
        r'|\[\d+\][:\s]+\S+\.(?:pdf|docx|xlsx|pptx|txt)'
        r'|\[\d+\]\s*\[)',
        answer,
        flags=re.MULTILINE,
    )[0].strip()

    # Normalise any [N: filename...] variants to bare [N]
    answer = re.sub(r'\[(\d+)[^\]]+\]', r'[\1]', answer)

    # -- Answer display --
    print('\n' + bold('=' * 60))
    print(bold('\U0001f4ac  ANSWER'))
    print(bold('=' * 60))
    for line in answer.split('\n'):
        if line.strip():
            print(textwrap.fill(line, width=80))
        else:
            print()

    # Deduplicate in prompt order so [N] numbers stay aligned
    seen, deduped = set(), []
    for r in retrieved:
        key = (r['source'], r['page'])
        if key not in seen:
            seen.add(key)
            deduped.append(r)

    # Match cited chunks by precise bracket regex
    cited      = []
    cited_keys = set()
    for i, r in enumerate(deduped):
        fname   = os.path.basename(r['source'])
        key     = (fname, r['page'])
        pattern = re.compile(rf'\[{i + 1}[,\]\s]')
        if pattern.search(answer):
            if key not in cited_keys:
                cited_keys.add(key)
                cited.append((i + 1, r))

    if cited:
        print('\n' + bold('-' * 60))
        print(bold('\U0001f4da  SOURCES CITED  ') + dim('(in order cited -- gaps = retrieved but not used)'))
        print(bold('-' * 60))

        # No re-sort -- preserve answer-citation order
        for orig_n, r in cited:
            d         = r['hybrid_score']
            authority = r.get('authority', 'opinion')
            auth_badge = magenta('\u2696\ufe0f  REGULATORY') if authority == 'regulatory' \
                         else blue('\U0001f4ac  OPINION   ')
            if d < 0.5:
                rel_badge = green('\U0001f7e2 HIGH  ')
            elif d < 0.65:
                rel_badge = yellow('\U0001f7e1 MEDIUM')
            else:
                rel_badge = red('\U0001f534 LOW   ')
            fname = os.path.basename(r['source'])
            print(f'  {auth_badge}  {rel_badge}  {bold(f"[{orig_n}]")}')
            print(f'         {bold(fname)}  --  page {r["page"]}')
            print(dim(f'         hybrid: {d:.3f}  |  rerank: {r["rerank_score"]:.2f}  |  semantic: {r["sem_score"]:.3f}  |  keyword: {r["kw_score"]:.3f}'))
            print()
    else:
        print('\n' + dim('\u26a0\ufe0f  No explicit citations found.'))
        print(dim('   Try qwen2.5:14b in Block 6, or increase TOP_K above.'))

    return answer


# Run!
answer = ask_question(MY_QUESTION, top_k=TOP_K, final_k=FINAL_K)



🧭 CDM Compass  --  "What skills does an experienced Clinical Data Manager of 25+ years need to transition to Data Science?"
   Hybrid: 70% semantic + 30% keyword  |  reranking top 5 of 10
   5 chunks after reranking
   Estimated prompt: ~1525 tokens
   Asking qwen2.5:14b...

💬  ANSWER
An experienced Clinical Data Manager with over 25 years of experience needs to
build upon their existing foundational competencies such as attention to detail,
therapeutic area knowledge, communication skills, systematic data review and
trending, project management, and design of data collection tools [1]. To
transition into a role as a Clinical Data Scientist, they must also acquire new
and refined skillsets including:

- Tailoring data collection systems, data review strategies, and training
requirements based on study needs [1].
- Understanding interdependencies with functions such as Clinical Operations,
Quality Assurance, and Procurement to manage vendors effectively [4].
- Demonstrating influential

## 💬 Block 8 -- Interactive Q&A Loop

Ask multiple questions without re-running any cells.
CDM Compass remembers the last `HISTORY_TURNS` questions and answers within this session, so follow-up questions like *'Then how does that apply to RBQM?'* work as expected.

Type your question and press Enter. Type `quit` to stop.


In [8]:
print("\U0001f9ed CDM Compass -- Interactive Q&A")
print(f"   Memory: last {HISTORY_TURNS} turns  |  type 'quit' to stop\n")

# Rolling conversation history: list of (question, answer) tuples
conversation_history = []

while True:
    question = input('\n\u2753 Your question: ').strip()

    if not question:
        print("   \u26a0\ufe0f  Please type a question.")
        continue

    if question.lower() in ("quit", "exit", "stop", "q"):
        print("\n\U0001f44b Goodbye!")
        break

    answer = ask_question(
        question,
        top_k=TOP_K,
        final_k=FINAL_K,
        history=conversation_history if HISTORY_TURNS > 0 else None,
    )

    # Store this turn; keep only the last HISTORY_TURNS
    conversation_history.append((question, answer))
    if len(conversation_history) > HISTORY_TURNS:
        conversation_history.pop(0)


🧭 CDM Compass -- Interactive Q&A
   Memory: last 3 turns  |  type 'quit' to stop


🧭 CDM Compass  --  "What can you tell me about data governance related to CDM or CDS?"
   Hybrid: 70% semantic + 30% keyword  |  reranking top 5 of 10
   5 chunks after reranking
   Estimated prompt: ~1226 tokens
   Asking qwen2.5:14b...

💬  ANSWER
The provided documents do not contain specific information regarding data
governance as it pertains to Clinical Data Management (CDM) or Clinical Data
Science (CDS). The excerpts focus on the evolution of roles, responsibilities in
managing clinical data, and the integration of technologies but do not delve
into detailed aspects of data governance. Therefore, insufficient context is
available to provide a comprehensive answer about data governance related to CDM
or CDS based solely on these documents [1], [2], [3].

------------------------------------------------------------
📚  SOURCES CITED  (in order cited -- gaps = retrieved but not used)
-----------------

---
## 📖 Quick Reference

### How CDM Compass finds answers
1. **Hybrid search** -- ChromaDB (meaning) + BM25 (keywords) retrieve `TOP_K` candidates
2. **Reranking** -- cross-encoder selects best `FINAL_K` by reading question + chunk together
3. **Compression** -- only the most relevant sentences per chunk reach the LLM
4. **Context-aware answer** -- last `HISTORY_TURNS` Q&A pairs included so follow-ups work

### Conversation memory
Block 8 keeps a rolling window of the last `HISTORY_TURNS` (default: 3) question/answer pairs.
The LLM sees this history as context -- useful for follow-up questions.
Set `HISTORY_TURNS = 0` in Block 7 to turn memory off.
A token estimate is printed before each answer -- if it exceeds `TOKEN_WARN_LIMIT` (default: 3000),
reduce `HISTORY_TURNS` or `FINAL_K`.

### Reading the sources list
Sources appear **in the order cited in the answer**. Gaps (e.g. [1],[2],[4]) mean those
chunks were retrieved but not used.

Each source shows:
- **Authority badge** -- ⚖️ REGULATORY (binding) or 💬 OPINION (recommendation)
- **Relevance badge** -- 🟢 HIGH / 🟡 MEDIUM / 🔴 LOW (hybrid score)
- **Scores** -- hybrid (lower=better), rerank (higher=better), semantic, keyword

| Badge | Hybrid score | Meaning |
|-------|-------------|---------|
| 🟢 HIGH | < 0.5 | Strong match |
| 🟡 MEDIUM | 0.5 - 0.65 | Partial match |
| 🔴 LOW | > 0.65 | Weak match |

### Tuning (all in Block 7)
| Setting | Default | Effect |
|---------|---------|--------|
| `TOP_K` | 10 | Candidates before reranking -- raise if answers miss known sources |
| `FINAL_K` | 5 | Chunks sent to LLM -- raise for broader answers |
| `HISTORY_TURNS` | 3 | Conversation memory window (0 = off) |
| `TOKEN_WARN_LIMIT` | 3000 | Warn if prompt exceeds this token estimate |
| `SEMANTIC_WEIGHT` | 0.7 | Raise for conceptual questions |
| `KEYWORD_WEIGHT` | 0.3 | Raise for specific terms, IDs, clause numbers |
| `CHUNK_SIZE` (Block 4) | 300 | Lower for more precise citations |

### Managing document versions
| Situation | What to do |
|-----------|------------|
| Updated document | Delete old, drop in new with same filename, re-run Blocks 3-5 |
| Keep both versions | Rename with version numbers, keep in same subfolder, re-run Blocks 3-5 |
